In [50]:
import pandas as pd

In [51]:
df = pd.read_csv("../data/raw/activities.csv")

df.shape


(2051, 103)

In [52]:
df["Activity Type"].value_counts()

Activity Type
Run                  1478
Weight Training       290
Ride                  157
Stand Up Paddling      54
Hike                   38
Swim                   21
Yoga                    6
Walk                    3
Workout                 2
Elliptical              1
Surfing                 1
Name: count, dtype: int64

In [53]:
df_runs = df[df["Activity Type"] == "Run"].copy()
df_runs.shape

(1478, 103)

In [54]:
columns_df = pd.DataFrame({
    "column_number": range(len(df_runs.columns)),
    "column_name": df_runs.columns
})

columns_df

,column_number,column_name
0,0,Activity ID
1,1,Activity Date
2,2,Activity Name
3,3,Activity Type
4,4,Activity Description
...,...,...
98,98,With Kid
99,99,Downhill Distance
100,100,Total Sets
101,101,Total Reps


In [55]:
columns_df.to_csv("../data/processed/columns_list.csv", index=False)

In [56]:
[col for col in df_runs.columns if ".1" in col]

['Elapsed Time.1',
 'Distance.1',
 'Max Heart Rate.1',
 'Relative Effort.1',
 'Commute.1']

In [57]:
df_runs[["Distance", "Distance.1"]].head(10)

,Distance,Distance.1
0,5.09,5090.9
2,7.05,7051.2
4,8.20,8199.7
5,7.13,7137.4
7,0.85,849.4
9,9.04,9040.6
10,6.75,6757.3
11,7.50,7501.9
13,5.11,5115.9
15,2.47,2475.6


In [58]:
core_columns = [
    "Activity Date",
    "Activity Name",
    "Distance",
    "Moving Time",
    "Average Speed",
    "Elevation Gain",
    "Average Grade",
    "Max Grade",
    "Average Heart Rate",
    "Max Heart Rate",
    "Relative Effort",
    "Average Cadence",
    "Calories",
    "Average Temperature",
    "Dirt Distance"
]

In [59]:
df_core = df_runs[core_columns].copy()
df_core.shape

(1478, 15)

In [60]:
df_core.isnull().sum()

Activity Date            0
Activity Name            0
Distance                 0
Moving Time              0
Average Speed            0
Elevation Gain           0
Average Grade            0
Max Grade                0
Average Heart Rate     118
Max Heart Rate         118
Relative Effort        118
Average Cadence         96
Calories               169
Average Temperature    405
Dirt Distance          235
dtype: int64

In [61]:
df_core.dtypes

Activity Date              str
Activity Name              str
Distance                   str
Moving Time            float64
Average Speed          float64
Elevation Gain         float64
Average Grade          float64
Max Grade              float64
Average Heart Rate     float64
Max Heart Rate         float64
Relative Effort        float64
Average Cadence        float64
Calories               float64
Average Temperature    float64
Dirt Distance          float64
dtype: object

In [62]:
df_core["Distance"] = pd.to_numeric(df_core["Distance"])

df_core["Activity Date"] = pd.to_datetime(
    df_core["Activity Date"],
    format="%b %d, %Y, %I:%M:%S %p"
)

df_core.dtypes

Activity Date          datetime64[us]
Activity Name                     str
Distance                      float64
Moving Time                   float64
Average Speed                 float64
Elevation Gain                float64
Average Grade                 float64
Max Grade                     float64
Average Heart Rate            float64
Max Heart Rate                float64
Relative Effort               float64
Average Cadence               float64
Calories                      float64
Average Temperature           float64
Dirt Distance                 float64
dtype: object

In [63]:
df_core["Moving Time Minutes"] = (df_core["Moving Time"] / 60).round(2)

df_core["Pace Min Per KM"] = (
    df_core["Moving Time Minutes"] / df_core["Distance"]
).round(2)

df_core[[
    "Distance",
    "Moving Time Minutes",
    "Pace Min Per KM"
]].head(10)

,Distance,Moving Time Minutes,Pace Min Per KM
0,5.09,38.18,7.50
2,7.05,58.30,8.27
4,8.20,52.92,6.45
5,7.13,40.92,5.74
7,0.85,7.90,9.29
9,9.04,74.22,8.21
10,6.75,43.83,6.49
11,7.50,40.48,5.40
13,5.11,30.43,5.95
15,2.47,16.68,6.75


In [64]:
df_core["Run Type"] = df_core["Activity Name"].str.contains(
    "trail",
    case=False,
    na=False
).map({
    True: "Trail",
    False: "Road"
})

df_core[["Activity Name", "Run Type"]].head(20)

,Activity Name,Run Type
0,Afternoon Trail Run,Trail
2,Afternoon Trail Run,Trail
4,Morning Run,Road
5,Lunch Run,Road
7,Evening Run,Road
9,Afternoon Run,Road
10,Afternoon Run,Road
11,Afternoon Run,Road
13,Afternoon Run,Road
15,Afternoon Run,Road


In [65]:
df_core["Year"] = df_core["Activity Date"].dt.year
df_core["Month"] = df_core["Activity Date"].dt.month

df_core[["Activity Date", "Year", "Month"]].head()

,Activity Date,Year,Month
0,2026-04-17 14:19:24,2026,4
2,2026-04-16 14:44:10,2026,4
4,2026-04-15 07:57:15,2026,4
5,2026-04-14 09:04:17,2026,4
7,2026-04-11 16:18:51,2026,4


In [66]:
df_core["Efficiency"] = (
    (df_core["Average Heart Rate"] * df_core["Pace Min Per KM"]) / 1000
).round(3)

df_core[[
    "Pace Min Per KM",
    "Average Heart Rate",
    "Efficiency"
]].head(10)

,Pace Min Per KM,Average Heart Rate,Efficiency
0,7.50,131.0,0.982
2,8.27,131.0,1.083
4,6.45,150.0,0.968
5,5.74,147.0,0.844
7,9.29,113.0,1.050
9,8.21,143.0,1.174
10,6.49,137.0,0.889
11,5.40,154.0,0.832
13,5.95,138.0,0.821
15,6.75,138.0,0.932


In [67]:
df_core[df_core["Distance"] < 2][[
    "Activity Date",
    "Distance",
    "Pace Min Per KM",
    "Average Heart Rate",
    "Run Type"
]].head(20)

,Activity Date,Distance,Pace Min Per KM,Average Heart Rate,Run Type
7,2026-04-11 16:18:51,0.85,9.29,113.0,Road
18,2026-03-29 06:04:53,1.92,6.53,128.0,Trail
23,2026-03-22 09:30:31,1.43,7.97,138.0,Road
60,2026-02-07 18:03:45,0.30,16.23,99.0,Road
62,2026-02-04 15:43:52,0.11,11.82,121.0,Road
93,2025-12-29 13:33:16,1.58,13.63,86.0,Trail
111,2025-12-07 07:18:47,1.01,8.07,128.0,Trail
162,2025-10-02 10:30:21,0.09,9.11,101.0,Trail
193,2025-08-30 13:10:04,1.36,11.54,108.0,Trail
227,2025-08-03 15:51:55,0.32,8.69,114.0,Road


In [68]:
(df_core["Distance"] < 2).sum()

np.int64(109)

In [69]:
df_core = df_core[df_core["Distance"] > 2].copy()
df_core.shape

(1366, 21)

In [70]:
df_core = df_core.dropna(subset=["Average Heart Rate"]).copy()

df_core.shape

(1286, 21)

In [71]:
df_core.isnull().sum()

Activity Date            0
Activity Name            0
Distance                 0
Moving Time              0
Average Speed            0
Elevation Gain           0
Average Grade            0
Max Grade                0
Average Heart Rate       0
Max Heart Rate           0
Relative Effort          0
Average Cadence         53
Calories                66
Average Temperature    288
Dirt Distance          132
Moving Time Minutes      0
Pace Min Per KM          0
Run Type                 0
Year                     0
Month                    0
Efficiency               0
dtype: int64

In [72]:
df_core[df_core["Pace Min Per KM"] > 15][[
    "Activity Date",
    "Distance",
    "Pace Min Per KM",
    "Average Heart Rate",
    "Elevation Gain",
    "Run Type"
]]

,Activity Date,Distance,Pace Min Per KM,Average Heart Rate,Elevation Gain,Run Type
37,2026-03-07 10:00:03,8.00,15.56,142.0,1068.0,Trail
102,2025-12-21 09:53:03,2.72,17.57,132.0,592.0,Trail
233,2025-07-30 16:15:53,4.00,21.91,135.0,215.0,Trail
287,2025-06-01 04:57:55,2.72,15.36,142.0,584.0,Trail
330,2025-04-19 14:26:24,2.47,18.10,123.0,19.0,Road
360,2025-03-15 09:53:15,11.25,15.24,147.0,1557.0,Road
431,2024-12-31 21:31:41,2.76,18.04,98.0,196.0,Trail
579,2024-08-03 10:01:58,7.50,21.50,96.0,864.0,Trail
614,2024-06-23 01:21:59,3.90,24.64,139.0,841.0,Trail
615,2024-06-22 08:11:17,10.78,37.24,118.0,1566.0,Trail


In [73]:
df_core = df_core[
    ~(
        (df_core["Run Type"] == "Road") &
        (df_core["Pace Min Per KM"] > 15)
    )
].copy()

df_core.shape

(1282, 21)

In [74]:
df_core[df_core["Pace Min Per KM"] > 15][[
    "Distance",
    "Pace Min Per KM",
    "Elevation Gain",
    "Run Type"
]]

,Distance,Pace Min Per KM,Elevation Gain,Run Type
37,8.00,15.56,1068.0,Trail
102,2.72,17.57,592.0,Trail
233,4.00,21.91,215.0,Trail
287,2.72,15.36,584.0,Trail
431,2.76,18.04,196.0,Trail
579,7.50,21.50,864.0,Trail
614,3.90,24.64,841.0,Trail
615,10.78,37.24,1566.0,Trail


In [75]:
df_core.groupby("Run Type")[[
    "Distance",
    "Pace Min Per KM",
    "Average Heart Rate",
    "Elevation Gain",
    "Efficiency"
]].mean().round(2)

,Distance,Pace Min Per KM,Average Heart Rate,Elevation Gain,Efficiency
Run Type,,,,,
Road,9.62,6.99,145.55,289.97,1.01
Trail,10.34,8.36,141.98,485.94,1.18


In [76]:
df_core["Run Type"].value_counts()

Run Type
Road     789
Trail    493
Name: count, dtype: int64

In [77]:
df_core.groupby("Run Type")[[
    "Distance",
    "Pace Min Per KM",
    "Average Heart Rate",
    "Elevation Gain",
    "Efficiency"
]].median().round(2)

,Distance,Pace Min Per KM,Average Heart Rate,Elevation Gain,Efficiency
Run Type,,,,,
Road,8.35,6.45,146.0,92.9,0.93
Trail,9.27,7.78,144.0,402.0,1.11


In [78]:
df_core.groupby("Year")["Pace Min Per KM"].median().round(2)

Year
2018    5.36
2019    5.70
2020    6.82
2021    6.55
2022    7.72
2023    7.26
2024    7.10
2025    6.94
2026    6.74
Name: Pace Min Per KM, dtype: float64

In [79]:
df_core.groupby(["Year", "Run Type"])[
    "Pace Min Per KM"
].median().round(2)

Year  Run Type
2018  Road        5.36
2019  Road        5.70
2020  Road        6.82
2021  Road        6.54
      Trail       9.37
2022  Road        7.56
      Trail       7.82
2023  Road        6.52
      Trail       7.77
2024  Road        5.96
      Trail       7.88
2025  Road        6.19
      Trail       7.72
2026  Road        5.98
      Trail       7.67
Name: Pace Min Per KM, dtype: float64

In [80]:
df_core.groupby(["Year", "Run Type"])[[
    "Pace Min Per KM", "Distance"]
].median().round(2)

Pace Min Per KM  Distance
Year Run Type                           
2018 Road                 5.36      9.69
2019 Road                 5.70      7.06
2020 Road                 6.82     12.09
2021 Road                 6.54      8.08
     Trail                9.37     23.88
2022 Road                 7.56      8.88
     Trail                7.82      7.40
2023 Road                 6.52      8.40
     Trail                7.77     10.00
2024 Road                 5.96      8.50
     Trail                7.88     10.01
2025 Road                 6.19      8.05
     Trail                7.72      9.00
2026 Road                 5.98      7.10
     Trail                7.67      8.78

In [81]:
df_core["Activity Name"] = (
    df_core["Activity Name"]
    .str.replace(r"[^a-zA-Z0-9 ]", "", regex=True)
)

In [87]:
df_core = df_core.fillna(0)

In [82]:
df_core["Activity Name"] = (
    df_core["Activity Name"]
    .str.encode("ascii", "ignore")
    .str.decode("ascii")
)

In [88]:
df_core.to_csv("../data/processed/runs_clean.csv", index=False)

In [84]:
df_core.head()

,Activity Date,Activity Name,Distance,Moving Time,Average Speed,Elevation Gain,Average Grade,Max Grade,Average Heart Rate,Max Heart Rate,...,Average Cadence,Calories,Average Temperature,Dirt Distance,Moving Time Minutes,Pace Min Per KM,Run Type,Year,Month,Efficiency
0,2026-04-17 14:19:24,Afternoon Trail Run,5.09,2291.0,2.222,187.0,-0.1,48.9,131.0,158.0,...,79.0,328.0,24.0,456.1,38.18,7.50,Trail,2026,4,0.982
2,2026-04-16 14:44:10,Afternoon Trail Run,7.05,3498.0,2.016,300.0,0.0,47.7,131.0,164.0,...,74.0,464.0,23.0,6165.7,58.30,8.27,Trail,2026,4,1.083
4,2026-04-15 07:57:15,Morning Run,8.20,3175.0,2.583,79.0,0.0,25.5,150.0,169.0,...,82.0,479.0,22.0,7489.0,52.92,6.45,Road,2026,4,0.968
5,2026-04-14 09:04:17,Lunch Run,7.13,2455.0,2.907,6.0,0.0,15.4,147.0,158.0,...,83.0,406.0,23.0,7063.6,40.92,5.74,Road,2026,4,0.844
9,2026-04-11 14:32:40,Afternoon Run,9.04,4453.0,2.030,273.0,-0.5,49.6,143.0,170.0,...,78.0,606.0,22.0,6762.5,74.22,8.21,Road,2026,4,1.174


In [85]:
check_df = pd.read_csv("../data/processed/runs_clean.csv")

check_df.shape

(1282, 21)